In [0]:
from pyspark.sql.functions import col, count, month, year, to_date, concat, lit, when, last_day, lpad
from pyspark.sql import Window
import json
import os

# Paths (Updated to avoid DBFS issues)
output_path = "/Volumes/prd_datascience_onlinejobpostings/volumes/onlinejobpostings/joep_keuzenkamp/solar_vacancies_data_2015_16"
checkpoint_path = "/Volumes/prd_datascience_onlinejobpostings/volumes/onlinejobpostings/joep_keuzenkamp/checkpoints.json"

# Convert DBFS path to local path for JSON handling
local_checkpoint_path = "/dbfs" + checkpoint_path.lstrip("dbfs:")

# Load all files at once (parallel processing)
df = spark.read.parquet("dbfs:/mnt/light/data/world_bank_global_postings/version=5/2025/07/09/18_00_44/*.parquet")

# Filter for US & jobs posted after 2017-01-01
df_us = df.filter((col('nation_name') == "United States") & (col('posted') >= "2015-01-01") & (col('posted') < "2017-01-01"))

# Define keywords for solar-related jobs
keywords = ["solar energy", "solar array", "pv system", "photovoltaic system", "solar powered", "solar power", "pv solar", "photovoltaic solar", "hybrid solar", "thermal pv", "thermal photovoltaic", "solar cook", "solar cooker", "solar furnace", "solar heater", "solar heating", "solar hot water", "solar irrigation", "solar lighting", "solar oven", "solar stove", "solar water", "photovoltaic cell", "pv cell", "photovoltaic efficiency", "pv efficiency", "photovoltaic energy", "pv energy", "photovoltaic power", "pv power", "pv panel", "photovoltaic panel", "rooftop solar", "solar battery", "solar cell", "solar hybrid", "solar panel", "solar pump", "solar pv", "solar photovoltaic", "solar pv system", "solar photovoltaic system", "solar thermal", "solar tower", "solar energies", "solar arrays", "pv systems", "photovoltaic systems", "solar powers", "solar cooks", "solar cookers", "solar furnaces", "solar heaters", "solar irrigations", "solar ovens", "solar stoves", "photovoltaic cells", "pv cells", "photovoltaic efficiencies", "pv efficiencies", "photovoltaic energies", "pv energies", "photovoltaic powers", "pv powers", "pv panels", "photovoltaic panels", "solar arrays", "solar batteries", "solar cells", "solar hybrids", "solar panels", "solar pumps", "solar pv systems", "solar photovoltaic systems", "solar thermals", "solar towers"]
regex_values = "|".join(keywords)

# Filter for solar-related jobs
solar_df = df_us.filter(col("body").rlike(regex_values))

new_solar_df = (
    solar_df.select("posted", "employment_type", "laa_admin_area_2", "min_years_experience", "isco_08_1")
    .withColumn("month", lpad(month(col("posted")).cast("string"), 2, "0"))
    .withColumn("year", year(col("posted")))
    .groupBy("year", "month", "laa_admin_area_2", "min_years_experience", "isco_08_1")
    .agg(
        count(when(col("employment_type") == 1, True)).alias("full_time_count"),
        count(when(col("employment_type") == 2, True)).alias("part_time_count"),
        count("*").alias("total_count")
    )
    .withColumnRenamed("laa_admin_area_2", "county_id")
    .withColumnRenamed("isco_08_1", "career_type")
    .withColumn("start_date", to_date(concat(col("year"), lit("-"), col("month"), lit("-01")), "yyyy-MM-dd"))
    .withColumn("end_date", last_day(col("start_date")))
)

# Append data as CSV (instead of Parquet)
new_solar_df.write.format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("sep", ",") \
    .save(output_path)

Now, we will check whether the code worked correctly. We check the structure of the dataframe, the first and final date of the vacancies data, and the last checkpoint reached.

In [0]:
# Load processed data
output_path = "/Volumes/prd_datascience_onlinejobpostings/volumes/onlinejobpostings/joep_keuzenkamp/solar_vacancies_data_2015_16"

df = spark.read.csv(output_path)

display(df)

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9
year,month,county_id,min_years_experience,career_type,full_time_count,part_time_count,total_count,start_date,end_date
2016,06,US6037,0,2,1,0,1,2016-06-01,2016-06-30
2015,07,US34005,null,2,0,3,3,2015-07-01,2015-07-31
2015,07,US39173,null,4,4,0,4,2015-07-01,2015-07-31
2015,01,US6071,null,7,1,0,1,2015-01-01,2015-01-31
2015,01,US6067,null,1,1,0,1,2015-01-01,2015-01-31
2015,01,US32003,5,2,5,0,5,2015-01-01,2015-01-31
2015,01,US6097,1,7,2,0,2,2015-01-01,2015-01-31
2015,07,US34015,null,2,1,1,2,2015-07-01,2015-07-31
2015,07,US15003,null,3,5,0,5,2015-07-01,2015-07-31
